# Notebook 4: robust DistilBERT experiments

**Research question:** Can a spam classifier trained on SMS messages generalize to emails?

This notebook fine-tunes DistilBERT under a locked robustness protocol. It adds validation-only hyperparameter selection, repeated training seeds, a controlled input-length experiment, an Enron class-count control, and confidence diagnostics. The final comparison uses the same frozen data partitions as the previous notebooks.

## Experiment design

- Split seed 42 remains fixed; training seeds never recreate the partitions.
- Hyperparameter tuning is restricted to the primary SMS-trained model. Three small candidates are compared with seeds 13, 42, and 73 using only SMS validation macro-F1.
- The reverse Enron-trained model is a secondary diagnostic and retains the predeclared reference configuration. This avoids expensive tuning unrelated to the primary question.
- After the SMS configuration is locked, both source models are trained from scratch with seeds 13, 42, 73, 101, and 137 and evaluated on all four test-domain pairs.
- Every checkpoint and the SMS candidate configuration are selected by source-validation macro-F1. The Enron configuration is fixed in advance, and the decision threshold stays at 0.5.
- The controlled length analysis changes only max_length across 64, 128, 256, and 512 for the SMS-trained model. Three seeds are used, and the 256-token results are reused from the locked experiment.
- The class-count control trains on one fixed Enron subset with exactly the SMS ham/spam training counts. Enron validation and both test sets remain unchanged.
- Mean and sample standard deviation measure training-seed variability. Stratified bootstrap intervals measure finite-test-sample uncertainty for each SMS-to-Enron run.

The test sets were inspected in the original project, so they are fixed confirmation benchmarks rather than pristine holdouts. Length results are a predeclared sensitivity analysis and never replace the locked 256-token headline result.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/kbozukov-coke/cross-domain-spam-detection.git'
PROJECT_NAME = 'cross-domain-spam-detection'
IS_KAGGLE = Path('/kaggle/working').exists()

if IS_KAGGLE:
    PROJECT_ROOT = Path('/kaggle/working') / PROJECT_NAME
    if not (PROJECT_ROOT / '.git').exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1', REPOSITORY_URL, str(PROJECT_ROOT)],
            check=True,
        )
    else:
        subprocess.run(
            ['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'],
            check=True,
        )
    subprocess.run(
        [
            sys.executable, '-m', 'pip', 'install', '-q',
            'transformers==4.57.6', 'accelerate>=1.10,<2',
        ],
        check=True,
    )
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import transformers
from sklearn.metrics import ConfusionMatrixDisplay

from src.controls import match_reference_class_counts
from src.data import load_prepared_splits, summarize_splits
from src.distilbert import run_distilbert_source_experiment
from src.evaluation import (
    METRIC_COLUMNS,
    aggregate_seed_metrics,
    calibration_table,
    stratified_bootstrap_ci,
)
from src.modeling import run_transfer_experiments
from src.protocol import (
    BOOTSTRAP_SEED,
    CONFIRMATION_TRAINING_SEEDS,
    DATA_SPLIT_SEED,
    FINAL_TRAINING_SEEDS,
    REFERENCE_TRAINING_SEED,
)

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_colwidth', 120)

print(f'Running in: {"Kaggle" if IS_KAGGLE else "local environment"}')
print(f'PyTorch: {torch.__version__}')
print(f'Transformers: {transformers.__version__}')
print(
    'GPU devices: '
    f'{[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}'
)

if IS_KAGGLE and not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU accelerator before running this notebook.')

## Load the prepared data

The cleaning and partitions are identical to the earlier notebooks. Test frames are loaded here but are not passed to the model runner during hyperparameter selection.

In [ ]:
splits, cleaning_audit = load_prepared_splits(random_state=DATA_SPLIT_SEED)
split_summary = summarize_splits(splits)
split_summary.loc[:, ['dataset', 'split', 'rows', 'ham', 'spam', 'spam_rate']]

## Locked configurations and artifact helper

The search changes one optimization choice at a time relative to the reference. Input length stays at 256 during HPO because length is reserved for its own controlled experiment. All models use two epochs, dynamic padding, balanced class weights, and the same batch settings.

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
HEADLINE_MAX_LENGTH = 256
LENGTH_VALUES = (64, 128, 256, 512)
TUNING_SEEDS = tuple(CONFIRMATION_TRAINING_SEEDS)
REPORTING_SEEDS = tuple(FINAL_TRAINING_SEEDS)
LENGTH_SEEDS = tuple(CONFIRMATION_TRAINING_SEEDS)

COMMON_TRAINING_CONFIG = {
    'model_name': MODEL_NAME,
    'epochs': 2,
    'per_device_train_batch_size': 8,
    'per_device_eval_batch_size': 8,
    'gradient_accumulation_steps': 1,
    'warmup_ratio': 0.1,
    'require_gpu': True,
    'verbose': False,
}

DISTILBERT_CANDIDATES = [
    {
        'config_id': 'reference',
        'candidate_order': 0,
        'learning_rate': 2e-5,
        'weight_decay': 0.01,
    },
    {
        'config_id': 'lower_lr',
        'candidate_order': 1,
        'learning_rate': 1e-5,
        'weight_decay': 0.01,
    },
    {
        'config_id': 'stronger_weight_decay',
        'candidate_order': 2,
        'learning_rate': 2e-5,
        'weight_decay': 0.10,
    },
]
REFERENCE_CONFIG = next(
    candidate for candidate in DISTILBERT_CANDIDATES
    if candidate['config_id'] == 'reference'
)

SOURCE_NOTEBOOK = 'https://www.kaggle.com/code/kaloyanbozukov/notebook4'
RESULTS_DIRECTORY = PROJECT_ROOT / 'results'
RESULTS_DIRECTORY.mkdir(parents=True, exist_ok=True)

def save_artifact(frame, filename):
    output = frame.copy()
    output['source_notebook'] = SOURCE_NOTEBOOK
    repository_path = RESULTS_DIRECTORY / filename
    output.to_csv(repository_path, index=False)
    if IS_KAGGLE:
        output.to_csv(Path('/kaggle/working') / filename, index=False)
    return output

pd.DataFrame(DISTILBERT_CANDIDATES)

In [ ]:
expected_fits = {
    'SMS validation-only tuning': len(DISTILBERT_CANDIDATES) * len(TUNING_SEEDS),
    'locked five-seed reporting': 2 * len(REPORTING_SEEDS),
    'additional length variants': (
        (len(LENGTH_VALUES) - 1) * len(LENGTH_SEEDS)
    ),
    'Enron count-matched control': len(REPORTING_SEEDS),
}
display(pd.Series(expected_fits, name='fits').to_frame())
print(f'Total sequential fits: {sum(expected_fits.values())}')
print('Expected dual-T4 runtime: approximately 1.5 to 2 hours.')

## Validation-only tuning for the primary SMS model

Each candidate is trained with three seeds and scored only on the SMS validation split. No evaluation frame is supplied to the runner, so this phase cannot produce test predictions. The winner is ranked by mean validation macro-F1, then lower sample standard deviation, then the declared candidate order.

In [ ]:
tuning_rows = []

for candidate in DISTILBERT_CANDIDATES:
    for training_seed in TUNING_SEEDS:
        print(
            f"Tuning SMS | {candidate['config_id']} | "
            f"seed {training_seed}"
        )
        history, validation_result, evaluations, details = (
            run_distilbert_source_experiment(
                splits['sms']['train'],
                splits['sms']['validation'],
                train_domain='sms',
                random_state=training_seed,
                train_max_length=HEADLINE_MAX_LENGTH,
                learning_rate=candidate['learning_rate'],
                weight_decay=candidate['weight_decay'],
                **COMMON_TRAINING_CONFIG,
            )
        )
        assert evaluations.empty
        assert not details

        row = {
            'train_domain': 'sms',
            'config_id': candidate['config_id'],
            'candidate_order': candidate['candidate_order'],
            'training_seed': training_seed,
            'learning_rate': candidate['learning_rate'],
            'weight_decay': candidate['weight_decay'],
            'train_max_length': HEADLINE_MAX_LENGTH,
            'minimum_validation_loss': history['eval_loss'].dropna().min(),
            'best_epoch': validation_result['best_epoch'],
            'epochs_trained': validation_result['epochs_trained'],
            'training_seconds': validation_result['training_seconds'],
            'parameters': validation_result['parameters'],
        }
        row.update(
            {
                f'validation_{metric}': validation_result[metric]
                for metric in METRIC_COLUMNS
            }
        )
        tuning_rows.append(row)

        del history, validation_result, evaluations, details

tuning_results = pd.DataFrame(tuning_rows)
assert len(tuning_results) == len(DISTILBERT_CANDIDATES) * len(TUNING_SEEDS)
assert not tuning_results.duplicated(
    ['train_domain', 'config_id', 'training_seed']
).any()

tuning_results = save_artifact(
    tuning_results,
    'distilbert_tuning_results.csv',
)
print(f'Finished {len(tuning_results)} validation-only tuning runs.')

In [ ]:
tuning_summary = (
    tuning_results
    .groupby(
        [
            'train_domain', 'config_id', 'candidate_order',
            'learning_rate', 'weight_decay', 'train_max_length',
        ],
        as_index=False,
    )
    .agg(
        validation_macro_f1_mean=('validation_macro_f1', 'mean'),
        validation_macro_f1_std=('validation_macro_f1', 'std'),
        validation_f1_mean=('validation_f1', 'mean'),
        validation_balanced_accuracy_mean=(
            'validation_balanced_accuracy', 'mean'
        ),
        validation_mcc_mean=('validation_mcc', 'mean'),
        validation_pr_auc_mean=('validation_pr_auc', 'mean'),
        validation_brier_score_mean=('validation_brier_score', 'mean'),
        validation_ece_mean=('validation_ece', 'mean'),
        minimum_validation_loss_mean=('minimum_validation_loss', 'mean'),
        training_seconds_mean=('training_seconds', 'mean'),
        n_seeds=('training_seed', 'nunique'),
    )
)

ranked_tuning = tuning_summary.sort_values(
    [
        'validation_macro_f1_mean',
        'validation_macro_f1_std',
        'candidate_order',
    ],
    ascending=[False, True, True],
    kind='stable',
)
selected_index = ranked_tuning.index[0]
tuning_summary['selected'] = False
tuning_summary.loc[selected_index, 'selected'] = True
assert tuning_summary['selected'].sum() == 1
assert tuning_summary['n_seeds'].eq(len(TUNING_SEEDS)).all()

selected_config_id = tuning_summary.loc[selected_index, 'config_id']
selected_sms_config = next(
    candidate for candidate in DISTILBERT_CANDIDATES
    if candidate['config_id'] == selected_config_id
)
locked_configs = {
    'sms': selected_sms_config,
    'enron': REFERENCE_CONFIG,
}

tuning_summary = save_artifact(
    tuning_summary,
    'distilbert_tuning_summary.csv',
)
display(
    tuning_summary.sort_values('candidate_order').round(4)
)
print(f"Locked SMS configuration: {selected_sms_config['config_id']}")
print('Locked Enron configuration: reference (predeclared diagnostic).')

## Locked five-seed test experiment

The configurations are now frozen. Each source model is retrained from scratch with all five reporting seeds and evaluated on both fixed test domains at 256 tokens. No best seed is selected. Only seed 42 is retained for learning curves, confusion matrices, calibration, and qualitative errors.

For each SMS-to-Enron seed, a stratified bootstrap reports a 95% F1 interval. These intervals quantify finite-test-sample uncertainty; the standard deviation across seeds quantifies training instability.

In [ ]:
test_frames = {
    domain: splits[domain]['test']
    for domain in ('sms', 'enron')
}

seed_result_frames = []
reference_histories = {}
reference_details = {}

for train_domain in ('sms', 'enron'):
    locked_config = locked_configs[train_domain]

    for training_seed in REPORTING_SEEDS:
        print(
            f"Locked {train_domain.upper()} | "
            f"{locked_config['config_id']} | seed {training_seed}"
        )
        retain_details = (
            train_domain == 'sms'
            or training_seed == REFERENCE_TRAINING_SEED
        )
        history, validation_result, run_results, details = (
            run_distilbert_source_experiment(
                splits[train_domain]['train'],
                splits[train_domain]['validation'],
                train_domain=train_domain,
                evaluation_frames=test_frames,
                evaluation_max_lengths=(HEADLINE_MAX_LENGTH,),
                detail_max_length=(
                    HEADLINE_MAX_LENGTH if retain_details else None
                ),
                random_state=training_seed,
                train_max_length=HEADLINE_MAX_LENGTH,
                learning_rate=locked_config['learning_rate'],
                weight_decay=locked_config['weight_decay'],
                **COMMON_TRAINING_CONFIG,
            )
        )
        run_results['config_id'] = locked_config['config_id']
        run_results['training_regime'] = 'full'
        run_results['f1_ci_low'] = np.nan
        run_results['f1_ci_high'] = np.nan
        run_results['bootstrap_repetitions'] = np.nan

        if train_domain == 'sms':
            primary_details = details[('enron', HEADLINE_MAX_LENGTH)]
            interval = stratified_bootstrap_ci(
                primary_details['label'],
                primary_details['spam_probability'],
                metric='f1',
                n_bootstrap=2_000,
                random_state=BOOTSTRAP_SEED,
            )
            primary_mask = run_results['test_domain'].eq('enron')
            run_results.loc[primary_mask, 'f1_ci_low'] = interval['ci_low']
            run_results.loc[primary_mask, 'f1_ci_high'] = interval['ci_high']
            run_results.loc[
                primary_mask, 'bootstrap_repetitions'
            ] = interval['n_bootstrap']

        if training_seed == REFERENCE_TRAINING_SEED:
            reference_histories[train_domain] = history.copy()
            for test_domain in ('sms', 'enron'):
                reference_details[(train_domain, test_domain)] = details[
                    (test_domain, HEADLINE_MAX_LENGTH)
                ].copy()

        seed_result_frames.append(run_results)
        del history, validation_result, run_results, details

distilbert_seed_results = pd.concat(seed_result_frames, ignore_index=True)
assert len(distilbert_seed_results) == 4 * len(REPORTING_SEEDS)
assert not distilbert_seed_results.duplicated(
    ['train_domain', 'test_domain', 'training_seed']
).any()
assert distilbert_seed_results['evaluation_max_length'].eq(
    HEADLINE_MAX_LENGTH
).all()

distilbert_seed_results = save_artifact(
    distilbert_seed_results,
    'distilbert_seed_results.csv',
)
print(f'Finished {len(distilbert_seed_results)} locked test evaluations.')

In [ ]:
_, distilbert_seed_summary = aggregate_seed_metrics(
    distilbert_seed_results,
    group_columns=[
        'model', 'config_id', 'train_domain', 'test_domain',
        'setting', 'training_regime', 'evaluation_max_length',
    ],
    metric_columns=list(METRIC_COLUMNS),
)
assert len(distilbert_seed_summary) == 4
assert distilbert_seed_summary['f1_count'].eq(
    len(REPORTING_SEEDS)
).all()

distilbert_seed_summary = save_artifact(
    distilbert_seed_summary,
    'distilbert_seed_summary.csv',
)

summary_columns = [
    'train_domain', 'test_domain',
    'f1_mean', 'f1_std',
    'macro_f1_mean', 'macro_f1_std',
    'balanced_accuracy_mean', 'balanced_accuracy_std',
    'mcc_mean', 'mcc_std',
    'roc_auc_mean', 'roc_auc_std',
    'pr_auc_mean', 'pr_auc_std',
    'brier_score_mean', 'brier_score_std',
    'log_loss_mean', 'log_loss_std',
    'ece_mean', 'ece_std',
]
display(distilbert_seed_summary.loc[:, summary_columns].round(3))

primary_intervals = distilbert_seed_results.query(
    "train_domain == 'sms' and test_domain == 'enron'"
).loc[
    :,
    [
        'training_seed', 'f1', 'f1_ci_low', 'f1_ci_high',
        'precision', 'recall', 'roc_auc', 'pr_auc',
    ],
]
display(primary_intervals.round(3))

## Representative learning curves

The plots show the predeclared reference seed 42 only. Aggregate conclusions come from all five seeds.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for axis, domain in zip(axes, ('sms', 'enron')):
    history = reference_histories[domain]
    train_rows = history.dropna(subset=['loss'])
    validation_rows = history.dropna(subset=['eval_loss'])
    axis.plot(
        train_rows['epoch'],
        train_rows['loss'],
        marker='o',
        label='train',
    )
    axis.plot(
        validation_rows['epoch'],
        validation_rows['eval_loss'],
        marker='o',
        label='validation',
    )
    axis.set_title(f'{domain.upper()} loss (seed 42)')
    axis.set_xlabel('Epoch')
    axis.set_ylabel('Cross-entropy')
    axis.legend()

plt.tight_layout()
plt.show()

## Training-seed variability and transfer gap

Dots show every locked run; the larger markers and bars show mean plus/minus one sample standard deviation. The SMS transfer gap is paired by training seed.

In [ ]:
plot_results = distilbert_seed_results.copy()
plot_results['experiment'] = (
    plot_results['train_domain'].str.upper()
    + ' -> '
    + plot_results['test_domain'].str.upper()
)
experiment_order = [
    'SMS -> SMS', 'SMS -> ENRON', 'ENRON -> SMS', 'ENRON -> ENRON'
]

plt.figure(figsize=(10, 5))
sns.stripplot(
    data=plot_results,
    x='experiment',
    y='f1',
    order=experiment_order,
    color='0.35',
    jitter=0.08,
    alpha=0.75,
)
sns.pointplot(
    data=plot_results,
    x='experiment',
    y='f1',
    order=experiment_order,
    errorbar='sd',
    capsize=0.15,
    color='#c44e52',
    markers='D',
    linestyles='none',
)
plt.title('DistilBERT F1 across five training seeds')
plt.xlabel('Train -> test domain')
plt.ylabel('Spam-class F1')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

sms_seed_f1 = (
    distilbert_seed_results
    .query("train_domain == 'sms'")
    .pivot(index='training_seed', columns='test_domain', values='f1')
    .rename(columns={'sms': 'sms_to_sms_f1', 'enron': 'sms_to_enron_f1'})
)
sms_seed_f1['transfer_gap'] = (
    sms_seed_f1['sms_to_sms_f1'] - sms_seed_f1['sms_to_enron_f1']
)
display(sms_seed_f1.round(3))
print(
    'Mean paired transfer gap: '
    f"{sms_seed_f1['transfer_gap'].mean():.3f} "
    f"± {sms_seed_f1['transfer_gap'].std(ddof=1):.3f}"
)

## Compare TF-IDF, TextCNN, and DistilBERT

The deterministic TF-IDF baseline is refitted once and has no training-seed standard deviation. TextCNN and DistilBERT are compared through their raw five-seed artifacts. Error bars for the neural models are sample standard deviations, not confidence intervals.

In [ ]:
_, baseline_results = run_transfer_experiments(
    splits,
    random_state=REFERENCE_TRAINING_SEED,
)
baseline_results = baseline_results.assign(
    model='TF-IDF + Logistic Regression'
)

textcnn_seed_path = RESULTS_DIRECTORY / 'textcnn_seed_results.csv'
textcnn_summary_path = RESULTS_DIRECTORY / 'textcnn_seed_summary.csv'
textcnn_seed_results = pd.read_csv(textcnn_seed_path)
textcnn_seed_summary = pd.read_csv(textcnn_summary_path)

expected_pairs = {
    ('sms', 'sms'), ('sms', 'enron'),
    ('enron', 'sms'), ('enron', 'enron'),
}
assert len(textcnn_seed_results) == 20
assert set(textcnn_seed_results['training_seed']) == set(REPORTING_SEEDS)
assert set(
    zip(
        textcnn_seed_results['train_domain'],
        textcnn_seed_results['test_domain'],
    )
) == expected_pairs
assert not textcnn_seed_results.duplicated(
    ['train_domain', 'test_domain', 'training_seed']
).any()
assert len(textcnn_seed_summary) == 4

baseline_summary = baseline_results.loc[
    :, ['model', 'train_domain', 'test_domain', 'f1']
].rename(columns={'f1': 'f1_mean'})
baseline_summary['f1_std'] = np.nan
baseline_summary['n_seeds'] = np.nan

textcnn_comparison = textcnn_seed_summary.loc[
    :, ['model', 'train_domain', 'test_domain', 'f1_mean', 'f1_std', 'f1_count']
].rename(columns={'f1_count': 'n_seeds'})
distilbert_comparison = distilbert_seed_summary.loc[
    :, ['model', 'train_domain', 'test_domain', 'f1_mean', 'f1_std', 'f1_count']
].rename(columns={'f1_count': 'n_seeds'})

comparison_summary = pd.concat(
    [baseline_summary, textcnn_comparison, distilbert_comparison],
    ignore_index=True,
)
comparison_summary['experiment'] = (
    comparison_summary['train_domain'].str.upper()
    + ' -> '
    + comparison_summary['test_domain'].str.upper()
)

def format_f1(row):
    if pd.isna(row['f1_std']):
        return f"{row['f1_mean']:.3f} (deterministic)"
    return f"{row['f1_mean']:.3f} ± {row['f1_std']:.3f}"

comparison_summary['reported_f1'] = comparison_summary.apply(
    format_f1,
    axis=1,
)
display(
    comparison_summary.pivot(
        index='experiment',
        columns='model',
        values='reported_f1',
    ).reindex(experiment_order)
)

neural_runs = pd.concat(
    [
        textcnn_seed_results.loc[
            :, ['model', 'train_domain', 'test_domain', 'training_seed', 'f1']
        ],
        distilbert_seed_results.loc[
            :, ['model', 'train_domain', 'test_domain', 'training_seed', 'f1']
        ],
    ],
    ignore_index=True,
)
neural_runs['experiment'] = (
    neural_runs['train_domain'].str.upper()
    + ' -> '
    + neural_runs['test_domain'].str.upper()
)

plt.figure(figsize=(11, 5))
axis = sns.pointplot(
    data=neural_runs,
    x='experiment',
    y='f1',
    hue='model',
    order=experiment_order,
    errorbar='sd',
    capsize=0.12,
    dodge=0.35,
    markers=['o', 's'],
)
baseline_positions = {
    experiment: position
    for position, experiment in enumerate(experiment_order)
}
for row in baseline_summary.itertuples(index=False):
    experiment = f'{row.train_domain.upper()} -> {row.test_domain.upper()}'
    axis.scatter(
        baseline_positions[experiment],
        row.f1_mean,
        color='black',
        marker='D',
        s=45,
        zorder=5,
    )
axis.scatter([], [], color='black', marker='D', label='TF-IDF deterministic')
axis.set_title('Mean F1 with training-seed SD for neural models')
axis.set_xlabel('Train -> test domain')
axis.set_ylabel('Spam-class F1')
axis.set_ylim(0, 1)
axis.legend(title='Model', loc='lower right')
plt.tight_layout()
plt.show()

## Controlled max-length sensitivity analysis

Only the primary SMS-trained model is varied. The selected optimizer configuration is fixed, and max_length is the sole changed setting. Lengths 64, 128, 256, and 512 are trained and evaluated consistently with seeds 13, 42, and 73. The existing 256-token runs are reused, so nine additional fits are required.

The test results describe sensitivity; they do not select a new headline length.

In [ ]:
length_result_frames = [
    distilbert_seed_results.query(
        "train_domain == 'sms' and training_seed in @LENGTH_SEEDS"
    ).copy()
]
length_result_frames[0]['experiment'] = 'max_length_sensitivity'

for max_length in LENGTH_VALUES:
    if max_length == HEADLINE_MAX_LENGTH:
        continue

    for training_seed in LENGTH_SEEDS:
        print(f'Length {max_length} | SMS | seed {training_seed}')
        history, validation_result, run_results, details = (
            run_distilbert_source_experiment(
                splits['sms']['train'],
                splits['sms']['validation'],
                train_domain='sms',
                evaluation_frames=test_frames,
                evaluation_max_lengths=(max_length,),
                random_state=training_seed,
                train_max_length=max_length,
                learning_rate=selected_sms_config['learning_rate'],
                weight_decay=selected_sms_config['weight_decay'],
                **COMMON_TRAINING_CONFIG,
            )
        )
        run_results['config_id'] = selected_sms_config['config_id']
        run_results['training_regime'] = 'full'
        run_results['experiment'] = 'max_length_sensitivity'
        length_result_frames.append(run_results)
        del history, validation_result, run_results, details

distilbert_length_results = pd.concat(
    length_result_frames,
    ignore_index=True,
)
assert len(distilbert_length_results) == (
    len(LENGTH_VALUES) * len(LENGTH_SEEDS) * 2
)
assert distilbert_length_results['train_max_length'].eq(
    distilbert_length_results['evaluation_max_length']
).all()
assert not distilbert_length_results.duplicated(
    [
        'train_domain', 'test_domain', 'training_seed',
        'train_max_length',
    ]
).any()

distilbert_length_results = save_artifact(
    distilbert_length_results,
    'distilbert_length_results.csv',
)
print(f'Finished {len(distilbert_length_results)} length evaluations.')

In [ ]:
_, distilbert_length_summary = aggregate_seed_metrics(
    distilbert_length_results,
    group_columns=[
        'model', 'config_id', 'train_domain', 'test_domain',
        'setting', 'experiment', 'train_max_length',
        'evaluation_max_length',
    ],
    metric_columns=list(METRIC_COLUMNS),
)
assert len(distilbert_length_summary) == len(LENGTH_VALUES) * 2
assert distilbert_length_summary['f1_count'].eq(
    len(LENGTH_SEEDS)
).all()

distilbert_length_summary = save_artifact(
    distilbert_length_summary,
    'distilbert_length_summary.csv',
)

display(
    distilbert_length_summary.loc[
        :,
        [
            'test_domain', 'train_max_length',
            'f1_mean', 'f1_std',
            'macro_f1_mean', 'macro_f1_std',
            'roc_auc_mean', 'pr_auc_mean',
        ],
    ].sort_values(['test_domain', 'train_max_length']).round(3)
)

plt.figure(figsize=(9, 5))
sns.lineplot(
    data=distilbert_length_results,
    x='train_max_length',
    y='f1',
    hue='test_domain',
    marker='o',
    errorbar='sd',
)
plt.axvline(
    HEADLINE_MAX_LENGTH,
    color='black',
    linestyle='--',
    linewidth=1,
    label='locked headline length',
)
plt.title('SMS-trained DistilBERT sensitivity to max_length')
plt.xlabel('Training and evaluation max_length')
plt.ylabel('Spam-class F1')
plt.xticks(LENGTH_VALUES)
plt.ylim(0, 1)
plt.legend(title='Test domain')
plt.tight_layout()
plt.show()

## Enron class-count-matched control

The Enron training split is sampled once to the exact SMS ham and spam counts using control seed 2026. The Enron validation split and both test sets are unchanged. The locked Enron reference configuration is trained with all five reporting seeds.

This removes the full Enron set's per-class count advantage. It is not a pure causal estimate of data volume: class prevalence, class weights, optimizer steps, and the sampled messages also change, and one fixed subset does not quantify subset-selection uncertainty.

In [ ]:
matched_enron_train = match_reference_class_counts(
    splits['enron']['train'],
    splits['sms']['train'],
)

training_count_comparison = pd.DataFrame(
    {
        'sms_full': splits['sms']['train']['label'].value_counts().sort_index(),
        'enron_full': splits['enron']['train']['label'].value_counts().sort_index(),
        'enron_count_matched': (
            matched_enron_train['label'].value_counts().sort_index()
        ),
    }
).rename(index={0: 'ham', 1: 'spam'})

assert training_count_comparison['sms_full'].equals(
    training_count_comparison['enron_count_matched']
)
display(training_count_comparison)

In [ ]:
count_control_frames = []

for training_seed in REPORTING_SEEDS:
    print(f'Count-matched ENRON | seed {training_seed}')
    history, validation_result, run_results, details = (
        run_distilbert_source_experiment(
            matched_enron_train,
            splits['enron']['validation'],
            train_domain='enron',
            evaluation_frames=test_frames,
            evaluation_max_lengths=(HEADLINE_MAX_LENGTH,),
            random_state=training_seed,
            train_max_length=HEADLINE_MAX_LENGTH,
            learning_rate=REFERENCE_CONFIG['learning_rate'],
            weight_decay=REFERENCE_CONFIG['weight_decay'],
            **COMMON_TRAINING_CONFIG,
        )
    )
    run_results['config_id'] = REFERENCE_CONFIG['config_id']
    run_results['training_regime'] = 'enron_count_matched_to_sms'
    count_control_frames.append(run_results)
    del history, validation_result, run_results, details

distilbert_count_control_results = pd.concat(
    count_control_frames,
    ignore_index=True,
)
assert len(distilbert_count_control_results) == 2 * len(REPORTING_SEEDS)
assert not distilbert_count_control_results.duplicated(
    ['train_domain', 'test_domain', 'training_seed']
).any()

distilbert_count_control_results = save_artifact(
    distilbert_count_control_results,
    'distilbert_count_control_results.csv',
)

In [ ]:
_, distilbert_count_control_summary = aggregate_seed_metrics(
    distilbert_count_control_results,
    group_columns=[
        'model', 'config_id', 'train_domain', 'test_domain',
        'setting', 'training_regime', 'evaluation_max_length',
    ],
    metric_columns=list(METRIC_COLUMNS),
)
assert len(distilbert_count_control_summary) == 2
assert distilbert_count_control_summary['f1_count'].eq(
    len(REPORTING_SEEDS)
).all()

distilbert_count_control_summary = save_artifact(
    distilbert_count_control_summary,
    'distilbert_count_control_summary.csv',
)

full_enron = distilbert_seed_results.query(
    "train_domain == 'enron'"
).loc[:, ['training_seed', 'test_domain', 'f1']].rename(
    columns={'f1': 'full_f1'}
)
matched_enron = distilbert_count_control_results.loc[
    :, ['training_seed', 'test_domain', 'f1']
].rename(columns={'f1': 'matched_f1'})
count_paired = full_enron.merge(
    matched_enron,
    on=['training_seed', 'test_domain'],
    validate='one_to_one',
)
count_paired['matched_minus_full'] = (
    count_paired['matched_f1'] - count_paired['full_f1']
)

display(
    distilbert_count_control_summary.loc[
        :,
        [
            'test_domain', 'f1_mean', 'f1_std',
            'macro_f1_mean', 'macro_f1_std',
            'balanced_accuracy_mean', 'mcc_mean',
        ],
    ].round(3)
)
display(count_paired.round(3))
display(
    count_paired.groupby('test_domain')['matched_minus_full']
    .agg(['mean', 'std', 'count'])
    .round(3)
)

## Confidence and calibration

Softmax spam probabilities are treated as confidence estimates, not automatically calibrated probabilities. Brier score, log loss, and ECE are already included in the five-seed summaries. The reliability diagram below is a representative diagnostic for the predeclared seed-42 SMS-to-Enron run.

In [ ]:
reference_primary = reference_details[('sms', 'enron')]
reliability = calibration_table(
    reference_primary['label'],
    reference_primary['spam_probability'],
    n_bins=10,
)
display(reliability.round(3))

plt.figure(figsize=(6, 5))
plt.plot([0, 1], [0, 1], linestyle='--', color='black', label='perfect calibration')
plt.plot(
    reliability['mean_probability'],
    reliability['observed_spam_rate'],
    marker='o',
    label='DistilBERT seed 42',
)
plt.xlabel('Mean predicted spam probability')
plt.ylabel('Observed spam rate')
plt.title('SMS -> Enron reliability diagram')
plt.legend()
plt.tight_layout()
plt.show()

## Representative confusion matrices and errors

These diagnostics use seed 42 only. The five-seed tables remain the basis for conclusions.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 9))

for axis, (train_domain, test_domain) in zip(
    axes.ravel(),
    [
        ('sms', 'sms'), ('sms', 'enron'),
        ('enron', 'sms'), ('enron', 'enron'),
    ],
):
    details = reference_details[(train_domain, test_domain)]
    ConfusionMatrixDisplay.from_predictions(
        details['label'],
        details['prediction'],
        display_labels=['ham', 'spam'],
        colorbar=False,
        ax=axis,
    )
    axis.set_title(
        f'Train {train_domain.upper()} -> '
        f'Test {test_domain.upper()} (seed 42)'
    )

plt.tight_layout()
plt.show()

In [ ]:
cross_domain_errors = reference_primary.loc[
    ~reference_primary['correct']
].copy()
cross_domain_errors['error_type'] = cross_domain_errors['label'].map(
    {0: 'false positive', 1: 'false negative'}
)
cross_domain_errors['confidence'] = np.maximum(
    cross_domain_errors['spam_probability'],
    1 - cross_domain_errors['spam_probability'],
)

display(
    cross_domain_errors['error_type']
    .value_counts()
    .rename('errors')
    .to_frame()
)
cross_domain_errors.sort_values(
    'confidence',
    ascending=False,
).loc[
    :, ['error_type', 'confidence', 'spam_probability', 'text']
].head(5)

## Final comparison

The summary below is generated from the executed artifacts. A small positive mean is not called robust when some seeds remain below the deterministic baseline. Length results are reported as sensitivity, not used to revise the locked configuration.

In [ ]:
def strict_summary_row(frame, train_domain, test_domain):
    rows = frame.loc[
        frame['train_domain'].eq(train_domain)
        & frame['test_domain'].eq(test_domain)
    ]
    if len(rows) != 1:
        raise ValueError(
            f'Expected one summary row for {train_domain}->{test_domain}; '
            f'found {len(rows)}.'
        )
    return rows.iloc[0]

baseline_primary = baseline_results.query(
    "train_domain == 'sms' and test_domain == 'enron'"
)['f1'].item()
textcnn_primary = strict_summary_row(
    textcnn_seed_summary,
    'sms',
    'enron',
)
distilbert_primary = strict_summary_row(
    distilbert_seed_summary,
    'sms',
    'enron',
)
primary_runs = distilbert_seed_results.query(
    "train_domain == 'sms' and test_domain == 'enron'"
)
seeds_above_baseline = int(primary_runs['f1'].gt(baseline_primary).sum())

length_primary = distilbert_length_summary.query(
    "test_domain == 'enron'"
).sort_values('train_max_length')
count_reverse_change = count_paired.query(
    "test_domain == 'sms'"
)['matched_minus_full']

print(f"Selected SMS config: {selected_sms_config['config_id']}")
print(
    'DistilBERT SMS -> Enron F1: '
    f"{distilbert_primary['f1_mean']:.3f} "
    f"± {distilbert_primary['f1_std']:.3f}"
)
print(
    'TextCNN SMS -> Enron F1:    '
    f"{textcnn_primary['f1_mean']:.3f} "
    f"± {textcnn_primary['f1_std']:.3f}"
)
print(f'TF-IDF SMS -> Enron F1:      {baseline_primary:.3f}')
print(
    'DistilBERT seeds above TF-IDF: '
    f'{seeds_above_baseline}/{len(REPORTING_SEEDS)}'
)
print(
    'DistilBERT paired transfer gap: '
    f"{sms_seed_f1['transfer_gap'].mean():.3f} "
    f"± {sms_seed_f1['transfer_gap'].std(ddof=1):.3f}"
)
print(
    'Length sensitivity on Enron test: '
    f"F1 {length_primary['f1_mean'].min():.3f} to "
    f"{length_primary['f1_mean'].max():.3f} "
    f'across {list(LENGTH_VALUES)} tokens'
)
print(
    'Count-matched change for Enron -> SMS: '
    f'{count_reverse_change.mean():+.3f} '
    f'± {count_reverse_change.std(ddof=1):.3f}'
)

if (
    distilbert_primary['f1_mean'] > baseline_primary
    and seeds_above_baseline == len(REPORTING_SEEDS)
):
    print(
        'Conclusion: DistilBERT has higher point F1 than the deterministic '
        'baseline in all five observed runs. This is seed consistency, not '
        'a statistical significance claim, and the transfer gap remains.'
    )
else:
    print(
        'Conclusion: DistilBERT does not show a consistent cross-domain point '
        'F1 advantage over the deterministic baseline across all five seeds.'
    )

## Export executed artifacts

Eight compact CSV files preserve every tuning run, all locked seeds, the length sensitivity experiment, and the count-matched control. Copies are written to the Kaggle Output directory for download.

In [ ]:
artifact_frames = {
    'distilbert_tuning_results.csv': tuning_results,
    'distilbert_tuning_summary.csv': tuning_summary,
    'distilbert_seed_results.csv': distilbert_seed_results,
    'distilbert_seed_summary.csv': distilbert_seed_summary,
    'distilbert_length_results.csv': distilbert_length_results,
    'distilbert_length_summary.csv': distilbert_length_summary,
    'distilbert_count_control_results.csv': distilbert_count_control_results,
    'distilbert_count_control_summary.csv': distilbert_count_control_summary,
}

for filename, frame in artifact_frames.items():
    artifact_frames[filename] = save_artifact(frame, filename)
    print(f'{filename}: {len(frame)} rows')

## Scope and limitations

- The hyperparameter search is deliberately small and applies only to the primary SMS-trained model. The reverse Enron model uses the predeclared reference configuration.
- Three tuning seeds are reused in the five-seed reporting stage; seed variability is descriptive rather than a fully independent HPO replication.
- Five training seeds provide a useful stability check but not a population-level guarantee.
- The length experiment uses three seeds and changes training and evaluation length together. Its test results are sensitivity evidence, not a selection rule.
- The count control uses one deterministic Enron subset. It matches exact class counts but also changes prevalence, class weights, training steps, and message composition; it is not a pure data-volume causal estimate.
- Balanced class weights and a fixed 0.5 threshold remain unchanged across domains. Target-domain threshold tuning is intentionally excluded.
- Softmax scores expose confidence and calibration behavior but are not guaranteed calibrated probabilities.
- The confirmation test sets were already inspected in the original project. External data would be required for a deployment claim.
- A separate zero-shot instruction-tuned generative experiment remains the next extension.